# Phase 4 — OGB datasets under the official protocol

Runs the ViRGo pipeline on OGB-sourced datasets, **structural only**, scored with **official OGB splits + metrics** (`eval_ogb.py`). Heavy lifting lives in `run_ogb.py`; this notebook only orchestrates.

- **Protocol:** train on official TRAIN only → select on **validation**, lock to `results/ogb_selection.json` (§6) → read **test once** (§7, guarded by the lock).
- **Datasets:** `ogbl_ddi` (link prediction, Hits@20; graph = training links only) · `ogbn_arxiv` (node classification, Accuracy; full citation graph, labels used for learning = training papers only). One notebook run per knob.
- **Existing routes only:** virtual graphs + deepwalk → nb2 zone, graphsage → nb3 zone, rows → `results/scoreboard.csv`.


In [1]:
"""Setup: repo root, config, knobs. Heavy lifting: run_ogb.py (ensure_virtual / embed / score / table / select / report)."""
import os, sys
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)                     # run everything from the repo root
sys.path[:0] = [str(Path.cwd()), str(Path.cwd() / "scripts")]

import numpy as np
import pandas as pd
import networkx as nx

import benchmark_config as cfg
import graph_io, make_ogb, results_io
from run_ogb import TASKS, ensure_virtual, embed, score, table, select, selection, report

DATASET = "ogbl_ddi"                                # <-- pick: "ogbl_ddi" (link pred, Hits@20) | "ogbn_arxiv" (node class, Accuracy)
K = 10                                              # locked by the Phase-3 ablations
SEEDS = cfg.VG_SEEDS                                # encoder seeds 42/43/44; the build seed stays REPRO["seed"]
SIMS = cfg.VG_SIMS                                  # psi / degree / centrality / original / hybrid
ENCODERS = ["graphsage_edge", "deepwalk"]
TASK_STR = TASKS[DATASET][1]
print(f"{DATASET} | task = {TASK_STR} | K={K} | seeds {SEEDS}")

/home/m-adam/miniconda/envs/i2v/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ogbl_ddi | task = link prediction (OGB official) | K=10 | seeds [42, 43, 44]


## 1 · Get the data

`make_ogb` downloads OGB once → edgelist + `.nodes` + labels/pairs + official split. Reused when present; `data.x` never loaded (structural-only). ddi's edgelist holds **training links only**.


In [2]:
"""OGB -> ViRGo files (edgelist + .nodes + labels/pairs + official split); reuse-or-create."""
info = make_ogb.ensure_ogb(DATASET)
info


{'edge_path': 'input/ogbl_ddi_train.edgelist',
 'pairs': 'splits/ogb/ogbl_ddi_pairs.npz',
 'eval': 'ogb',
 'task': 'linkpred'}

## 2 · Load + check


In [3]:
"""One shared graph definition; check() prints the facts every downstream stage assumes."""
EDGELIST = str(cfg.DATASETS[DATASET]["edgelist"])
G = graph_io.load_graph(EDGELIST)                   # .nodes sidecar restores isolated nodes
props = graph_io.check(G, DATASET)


ogbl_ddi: 4267 nodes, 1067911 edges, max degree 2234


## 3 · Virtual graphs

Five variants, build-or-reuse; each build logs a row to `results/graph_health.csv`.


In [4]:
"""Build-or-reuse the five graph variants (top-K structural neighbors)."""
VG = {sim: ensure_virtual(G, DATASET, K, sim) for sim in SIMS}
for sim, V in VG.items():
    print(f"{sim:>10}: {V.number_of_nodes()} nodes / {V.number_of_edges()} edges")


       psi: 4267 nodes / 27042 edges
    degree: 4267 nodes / 27109 edges
centrality: 4267 nodes / 24886 edges
  original: 4267 nodes / 1067911 edges
    hybrid: 4267 nodes / 1088727 edges


## 4 · Train embeddings

Every (variant × encoder × seed), trained on the official TRAIN graph only — unsupervised, labels never touch training; reuse-or-create. GraphSAGE → nb3 zone, DeepWalk → nb2 zone. Restarted kernel ⇒ rerun §0–§4 (instant reuse).


In [5]:
"""Train-or-reuse every (variant x encoder x seed) embedding on the official TRAIN graph."""
EMB = {(sim, enc, s): embed(G, VG[sim], DATASET, K, sim, enc, s, EDGELIST)
       for sim in SIMS for enc in ENCODERS for s in SEEDS}
print(f"{len(EMB)} embeddings ready")


reuse  psi graphsage_edge seed 42 -> graphsage_edge_s42.emb
reuse  psi graphsage_edge seed 43 -> graphsage_edge_s43.emb
reuse  psi graphsage_edge seed 44 -> graphsage_edge_s44.emb
reuse  psi deepwalk seed 42 -> deepwalk_s42.emb
reuse  psi deepwalk seed 43 -> deepwalk_s43.emb
reuse  psi deepwalk seed 44 -> deepwalk_s44.emb
reuse  degree graphsage_edge seed 42 -> graphsage_edge_s42.emb
reuse  degree graphsage_edge seed 43 -> graphsage_edge_s43.emb
reuse  degree graphsage_edge seed 44 -> graphsage_edge_s44.emb
reuse  degree deepwalk seed 42 -> deepwalk_s42.emb
reuse  degree deepwalk seed 43 -> deepwalk_s43.emb
reuse  degree deepwalk seed 44 -> deepwalk_s44.emb
reuse  centrality graphsage_edge seed 42 -> graphsage_edge_s42.emb
reuse  centrality graphsage_edge seed 43 -> graphsage_edge_s43.emb
reuse  centrality graphsage_edge seed 44 -> graphsage_edge_s44.emb
reuse  centrality deepwalk seed 42 -> deepwalk_s42.emb
reuse  centrality deepwalk seed 43 -> deepwalk_s43.emb
reuse  centrality deepw

## 5 · Validation scores

The selection metric → `results/scoreboard.csv` as `valid_acc` / `valid_hits@20`.

- **ddi:** pairs are scored by a **trained decoder** (hadamard → MLP), fitted per config on **training edges only** — same decoder + seed for every variant. ~17 s per config, so this cell takes ~15 min.
- **arxiv:** the probe learns from **training-paper labels only**.

In [6]:
"""Score every embedding on the OFFICIAL VALIDATION split -> scoreboard rows, then the graph x encoder table."""
for sim in SIMS:
    for enc in ENCODERS:
        per = {}
        for s in SEEDS:
            for m, v in score(DATASET, str(EMB[(sim, enc, s)]), "valid", s).items():
                per.setdefault(m, []).append(v)
        for m, vals in per.items():
            results_io.record_score(DATASET, enc, sim, K, TASK_STR, SEEDS, vals, metric=m)
        print(f"{sim:>10} {enc:>14} | " + " ".join(f"{m}={np.mean(v):.4f}" for m, v in per.items()), flush=True)

table(DATASET, "valid", better=True)                # one row per graph variant, one column per encoder

/home/m-adam/miniconda/envs/i2v/lib/python3.12/site-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


    decoder epoch   0 | loss 1.3789
    decoder epoch  10 | loss 1.1200
    decoder epoch  20 | loss 0.9937
    decoder epoch  30 | loss 0.8800
    decoder epoch  40 | loss 0.8236
    decoder epoch  49 | loss 0.7774
    decoder epoch   0 | loss 1.3768
    decoder epoch  10 | loss 1.1116
    decoder epoch  20 | loss 0.9606
    decoder epoch  30 | loss 0.8446
    decoder epoch  40 | loss 0.7836
    decoder epoch  49 | loss 0.7746
    decoder epoch   0 | loss 1.3785
    decoder epoch  10 | loss 1.1278
    decoder epoch  20 | loss 0.9723
    decoder epoch  30 | loss 0.8485
    decoder epoch  40 | loss 0.7933
    decoder epoch  49 | loss 0.7727
       psi graphsage_edge | valid_hits@20=0.0223
    decoder epoch   0 | loss 1.3546
    decoder epoch  10 | loss 1.1197
    decoder epoch  20 | loss 1.0122
    decoder epoch  30 | loss 0.9729
    decoder epoch  40 | loss 0.9554
    decoder epoch  49 | loss 0.9459
    decoder epoch   0 | loss 1.3580
    decoder epoch  10 | loss 1.1294
    decoder epo

,GraphSAGE Hits@20,DeepWalk Hits@20,Better encoder
Graph version,,,
Ψ,0.0223,0.0071,GraphSAGE
Degree,0.0213,0.0071,GraphSAGE
Centrality,0.0309,0.0077,GraphSAGE
Original,0.0385,0.0772,DeepWalk
Hybrid,0.0385,0.1038,DeepWalk — best overall


## 6 · Selection (validation) — locks the winner

Winner picked on validation and **saved to `results/ogb_selection.json`**. §7 refuses without it; once test rows exist, re-selection refuses. Test never changes the winner.


In [7]:
"""Validation table + winner, LOCKED to results/ogb_selection.json."""
sel = select(DATASET)


  Encoder      Graph  Validation Hits@20  Std. deviation
 DeepWalk     Hybrid              0.1038          0.0040
 DeepWalk   Original              0.0772          0.0082
GraphSAGE   Original              0.0385          0.0032
GraphSAGE     Hybrid              0.0385          0.0029
GraphSAGE Centrality              0.0309          0.0050
GraphSAGE          Ψ              0.0223          0.0104
GraphSAGE     Degree              0.0213          0.0099
 DeepWalk Centrality              0.0077          0.0009
 DeepWalk     Degree              0.0071          0.0011
 DeepWalk          Ψ              0.0071          0.0007

Validation winner: Hybrid graph + DeepWalk, Hits@20 = 0.1038 ± 0.0040   (locked -> /home/m-adam/identity2vec/results/ogb_selection.json)


## 7 · Final test — run ONCE, after §6 is locked

Guarded: reads the §6 lock and refuses without it. Reuses the saved embeddings; retrains nothing except the ddi decoder (deterministic, seeded, train-edges only).

In [8]:
"""ONE test read -> test_* scoreboard rows (arxiv also gets weighted/macro F1 secondaries), then the test table."""
locked = selection(DATASET)                          # the saved §6 choice; asserts the lock exists BEFORE any test score
for sim in SIMS:
    for enc in ENCODERS:
        per = {}
        for s in SEEDS:
            for m, v in score(DATASET, str(EMB[(sim, enc, s)]), "test", s).items():
                per.setdefault(m, []).append(v)
        for m, vals in per.items():
            results_io.record_score(DATASET, enc, sim, K, TASK_STR, SEEDS, vals, metric=m)
        print(f"{sim:>10} {enc:>14} | " + " ".join(f"{m}={np.mean(v):.4f}" for m, v in per.items()), flush=True)

table(DATASET, "test")                               # reported, never re-selected from

    decoder epoch   0 | loss 1.3789
    decoder epoch  10 | loss 1.1200
    decoder epoch  20 | loss 0.9937
    decoder epoch  30 | loss 0.8800
    decoder epoch  40 | loss 0.8236
    decoder epoch  49 | loss 0.7774
    decoder epoch   0 | loss 1.3768
    decoder epoch  10 | loss 1.1116
    decoder epoch  20 | loss 0.9606
    decoder epoch  30 | loss 0.8446
    decoder epoch  40 | loss 0.7836
    decoder epoch  49 | loss 0.7746
    decoder epoch   0 | loss 1.3785
    decoder epoch  10 | loss 1.1278
    decoder epoch  20 | loss 0.9723
    decoder epoch  30 | loss 0.8485
    decoder epoch  40 | loss 0.7933
    decoder epoch  49 | loss 0.7727
       psi graphsage_edge | test_hits@20=0.0412
    decoder epoch   0 | loss 1.3546
    decoder epoch  10 | loss 1.1197
    decoder epoch  20 | loss 1.0122
    decoder epoch  30 | loss 0.9729
    decoder epoch  40 | loss 0.9554
    decoder epoch  49 | loss 0.9459
    decoder epoch   0 | loss 1.3580
    decoder epoch  10 | loss 1.1294
    decoder epoc

,GraphSAGE test Hits@20,DeepWalk test Hits@20
Graph,,
Ψ,0.0412,0.0081
Degree,0.0265,0.0080
Centrality,0.0514,0.0100
Original,0.0102,0.0378
Hybrid,0.0117,0.0533


## 8 · Results

Locked validation winner → its test score → both tables → the one-line conclusion. The winner is read from the §6 lock, never re-picked from the test table.

In [9]:
"""Locked winner + its test score, both tables, and the conclusion line."""
report(DATASET)

Validation winner (locked 2026-07-23T06:42:40): Hybrid graph + DeepWalk, Hits@20 = 0.1038 ± 0.0040

Validation
               GraphSAGE Hits@20  DeepWalk Hits@20           Better encoder
Graph version                                                              
Ψ                         0.0223            0.0071                GraphSAGE
Degree                    0.0213            0.0071                GraphSAGE
Centrality                0.0309            0.0077                GraphSAGE
Original                  0.0385            0.0772                 DeepWalk
Hybrid                    0.0385            0.1038  DeepWalk — best overall

Test
            GraphSAGE test Hits@20  DeepWalk test Hits@20
Graph                                                    
Ψ                           0.0412                 0.0081
Degree                      0.0265                 0.0080
Centrality                  0.0514                 0.0100
Original                    0.0102                 0.0378
Hyb